In [2]:
import yfinance as yf
import pandas as pd
import os

In [3]:
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../docs", exist_ok=True)

In [4]:
ticker = "INR=X"

df = yf.download(
    tickers=ticker,
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=True
)

print(f"Downloaded {len(df)} rows")

[*********************100%***********************]  1 of 1 completed

Downloaded 1300 rows


In [5]:
save_path = "../data/raw/USDINR_daily_raw.csv"
df.to_csv(save_path)
print(f"Saved to: {save_path}")

Saved to: ../data/raw/USDINR_daily_raw.csv


In [6]:
print("=== SHAPE ===")
print(df.shape)

print("\n=== FIRST 5 ROWS ===")
print(df.head())

print("\n=== LAST 5 ROWS ===")
print(df.tail())

print("\n=== NULL CHECK ===")
null_counts = df.isnull().sum()
print(null_counts)

if null_counts.sum() == 0:
    print("\n✅ No nulls found")
else:
    print(f"\n⚠️ Total nulls: {null_counts.sum()} — do NOT fix today, just note it")

=== SHAPE ===
(1300, 5)

=== FIRST 5 ROWS ===
Price           Close       High        Low       Open Volume
Ticker          INR=X      INR=X      INR=X      INR=X  INR=X
Date                                                         
2021-03-04  72.880699  73.108002  72.612198  72.880699      0
2021-03-05  73.131798  73.311203  72.735001  73.131798      0
2021-03-08  73.133400  73.415298  72.940002  73.133400      0
2021-03-09  73.330002  73.386299  72.885597  73.330399      0
2021-03-10  72.775902  73.078003  72.764900  72.775803      0

=== LAST 5 ROWS ===
Price           Close       High        Low       Open Volume
Ticker          INR=X      INR=X      INR=X      INR=X  INR=X
Date                                                         
2026-02-26  90.950996  91.066002  90.808998  90.951202      0
2026-02-27  91.007401  91.132301  90.894302  91.007401      0
2026-03-02  91.079597  91.778503  91.075798  91.075798      0
2026-03-03  91.563400  92.421303  91.560600  91.563004      0
202

In [7]:
ticker = "BRL=X"

df2 = yf.download(
    tickers=ticker,
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=True
)

print(f"Downloaded {len(df)} rows")

[*********************100%***********************]  1 of 1 completed

Downloaded 1300 rows


In [11]:
save_path = "../data/raw/USDBRL_daily_raw.csv"
df2.to_csv(save_path)
print(f"Saved to: {save_path}")

Saved to: ../data/raw/USDBRL_daily_raw.csv


In [31]:
# Read CSV, skipping the multi-level header rows from yfinance
df_inr = pd.read_csv("../data/raw/USDINR_daily_raw.csv", skiprows=[1, 2])
df_brl = pd.read_csv("../data/raw/USDBRL_daily_raw.csv", skiprows=[1, 2])

# Rename columns: "Price" contains dates, "Close" contains prices
df_inr = df_inr.rename(columns={"Price": "Date", "Close": "USDINR_Close"})
df_brl = df_brl.rename(columns={"Price": "Date", "Close": "USDBRL_Close"})

# Keep only what we need
df_inr = df_inr[["Date", "USDINR_Close"]]
df_brl = df_brl[["Date", "USDBRL_Close"]]

print("INR shape:", df_inr.shape)
print("BRL shape:", df_brl.shape)
print("\nINR head:\n", df_inr.head(3))
print("\nBRL head:\n", df_brl.head(3))

INR shape: (1300, 2)
BRL shape: (1300, 2)

INR head:
          Date  USDINR_Close
0  2021-03-04     72.880699
1  2021-03-05     73.131798
2  2021-03-08     73.133400

BRL head:
          Date  USDBRL_Close
0  2021-03-04        5.6175
1  2021-03-05        5.6683
2  2021-03-08        5.6895


In [33]:
# Merge explicitly on the Date column — never on index
df_merged = pd.merge(df_inr, df_brl, on="Date", how="inner")

print(f"Merged shape: {df_merged.shape}")
print(f"Date range: {df_merged['Date'].min()} → {df_merged['Date'].max()}")
print(f"\nRows dropped due to date mismatch: {len(df_inr) - len(df_merged)}")
print(df_merged.head())

Merged shape: (1300, 3)
Date range: 2021-03-04 → 2026-03-04

Rows dropped due to date mismatch: 0
         Date  USDINR_Close  USDBRL_Close
0  2021-03-04     72.880699        5.6175
1  2021-03-05     73.131798        5.6683
2  2021-03-08     73.133400        5.6895
3  2021-03-09     73.330002        5.8746
4  2021-03-10     72.775902        5.8015


In [34]:
# Formula: how many INR does 1 BRL cost, routing through USD
# If 1 USD = 83.5 INR and 1 USD = 5.0 BRL
# Then 1 BRL = 83.5 / 5.0 = 16.7 INR

df_merged["INR_BRL_synthetic"] = df_merged["USDINR_Close"] / df_merged["USDBRL_Close"]

print("Cross rate formula: USDINR_Close / USDBRL_Close")
print(df_merged[["Date", "USDINR_Close", "USDBRL_Close", "INR_BRL_synthetic"]].head(10))

Cross rate formula: USDINR_Close / USDBRL_Close
         Date  USDINR_Close  USDBRL_Close  INR_BRL_synthetic
0  2021-03-04     72.880699      5.617500          12.973868
1  2021-03-05     73.131798      5.668300          12.901892
2  2021-03-08     73.133400      5.689500          12.854100
3  2021-03-09     73.330002      5.874600          12.482552
4  2021-03-10     72.775902      5.801500          12.544325
5  2021-03-11     72.706001      5.671100          12.820440
6  2021-03-12     72.639000      5.534800          13.124051
7  2021-03-15     72.682999      5.495156          13.226740
8  2021-03-16     72.515297      5.614500          12.915718
9  2021-03-17     72.515602      5.623400          12.895330


In [35]:
save_path_cross = "../data/raw/INRBRL_synthetic_raw.csv"
df_merged.to_csv(save_path_cross, index=False)
print(f"Saved to: {save_path_cross}")
print(f"Total rows saved: {len(df_merged)}")

Saved to: ../data/raw/INRBRL_synthetic_raw.csv
Total rows saved: 1300


In [37]:
cross = df_merged["INR_BRL_synthetic"]

print("=== INR/BRL SYNTHETIC CROSS RATE STATS ===")
print(f"  Min  : {cross.min():.4f} INR per BRL")
print(f"  Max  : {cross.max():.4f} INR per BRL")
print(f"  Mean : {cross.mean():.4f} INR per BRL")
print(f"  Std  : {cross.std():.4f}")

print("\n=== ECONOMIC SENSE CHECK ===")
if 10 <= cross.mean() <= 25:
    print(f"✅ Mean {cross.mean():.2f} is within expected range (10-25 INR per BRL)")
else:
    print(f"⚠️ Mean {cross.mean():.2f} is OUTSIDE expected range — note it, don't fix today")

# Check for any zeros or negatives (would break the formula)
bad_rows = df_merged[df_merged["USDBRL_Close"] <= 0]
print(f"\nRows with zero/negative USDBRL_Close: {len(bad_rows)}")
if len(bad_rows) > 0:
    print(bad_rows)
else:
    print("✅ No division-by-zero risk found")

=== INR/BRL SYNTHETIC CROSS RATE STATS ===
  Min  : 12.4826 INR per BRL
  Max  : 17.7557 INR per BRL
  Mean : 15.4738 INR per BRL
  Std  : 1.1641

=== ECONOMIC SENSE CHECK ===
✅ Mean 15.47 is within expected range (10-25 INR per BRL)

Rows with zero/negative USDBRL_Close: 0
✅ No division-by-zero risk found


In [38]:
print("Downloading hourly USD/INR (this may take a moment)...")

df_inr_hourly = yf.download(
    tickers="INR=X",
    period="2y",
    interval="1h",
    auto_adjust=True,
    progress=True
)

print(f"Hourly USD/INR rows: {len(df_inr_hourly)}")

save_path = "../data/raw/USDINR_hourly_raw.csv"
df_inr_hourly.to_csv(save_path)
print(f"Saved to: {save_path}")

[*********************100%***********************]  1 of 1 completed

Hourly USD/INR rows: 10547
Saved to: ../data/raw/USDINR_hourly_raw.csv


In [39]:
print("Downloading hourly USD/BRL...")

df_brl_hourly = yf.download(
    tickers="BRL=X",
    period="2y",
    interval="1h",
    auto_adjust=True,
    progress=True
)

print(f"Hourly USD/BRL rows: {len(df_brl_hourly)}")

save_path = "../data/raw/USDBRL_hourly_raw.csv"
df_brl_hourly.to_csv(save_path)
print(f"Saved to: {save_path}")

[*********************100%***********************]  1 of 1 completed

Hourly USD/BRL rows: 8414
Saved to: ../data/raw/USDBRL_hourly_raw.csv


## FRED Macro: India Repo Rate
#### --> FRED stands for Federal Reserve Economic Data,It’s a massive online database.
#### --> The repo rate is the key policy interest rate set by the Reserve Bank of India (RBI).
#####  1. It’s the rate at which RBI lends money to commercial banks when they need short-term funds.
#####   2. By changing the repo rate, RBI influences:
#####      i. Inflation → Raising the repo rate makes borrowing costlier, reducing demand and cooling inflation.
#####      ii. Growth → Lowering the repo rate makes loans cheaper, encouraging spending and investment

In [ ]:
import pandas_datareader.data as web  # pandas_datareader --> library that can fetch financial/economic data from sources like FRED, Yahoo, etc
from datetime import datetime         # to handle start and end dates.

FRED_API_KEY = "4603ba4e1d5e49992331f744a18b2837"  # My API Key

try:
    start = datetime(2019, 1, 1)  # 2019 Jan 1
    end   = datetime.today()

    df_repo = web.DataReader(
        "INDIRLTLT01STM",   # is the series code in FRED for India’s long-term interest rate (monthly)
        "fred",             # tells pandas_datareader to fetch data from the FRED database
        start,
        end,
        api_key=FRED_API_KEY
    )

    df_repo.index.name = "Date"              # Renames the index to "Date"
    df_repo.columns   = ["India_Repo_Rate"]  # Renames the column to "India_Repo_Rate"

    save_path = "../data/raw/india_repo_rate_raw.csv"
    df_repo.to_csv(save_path)

    print(f"India Repo Rate rows: {len(df_repo)}")
    print(df_repo.head())
    print(f"Saved to: {save_path}")

except Exception as e:
    print(f"⚠️  FRED fetch failed: {e}")
    print("Skipping for today — note this in your daily log.")
    df_repo = None

India Repo Rate rows: 85
            India_Repo_Rate
Date                       
2019-01-01             7.35
2019-02-01             7.43
2019-03-01             7.41
2019-04-01             7.42
2019-05-01             7.33
Saved to: ../data/raw/india_repo_rate_raw.csv


In [43]:
import os

# Helper to extract info from a CSV safely
def get_file_info(filepath, date_col=None):
    try:
        df = pd.read_csv(filepath)

        # Try to detect date column automatically
        if date_col is None:
            for col in ["Date", "DATE", "Datetime", "datetime", "index"]:
                if col in df.columns:
                    date_col = col
                    break

        if date_col and date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            start = df[date_col].min().date()
            end   = df[date_col].max().date()
        else:
            start, end = "N/A", "N/A"

        nulls = df.isnull().sum().sum()
        size_kb = os.path.getsize(filepath) / 1024

        return {
            "rows"    : len(df),
            "cols"    : len(df.columns),
            "start"   : start,
            "end"     : end,
            "nulls"   : int(nulls),
            "size_kb" : round(size_kb, 1)
        }
    except Exception as e:
        return {"rows": "ERROR", "cols": str(e), "start": "", "end": "", "nulls": "", "size_kb": ""}

# All 5 files
files = {
    "USDINR daily"    : "../data/raw/USDINR_daily_raw.csv",
    "USDBRL daily"    : "../data/raw/USDBRL_daily_raw.csv",
    "INRBRL synthetic": "../data/raw/INRBRL_synthetic_raw.csv",
    "USDINR hourly"   : "../data/raw/USDINR_hourly_raw.csv",
    "USDBRL hourly"   : "../data/raw/USDBRL_hourly_raw.csv",
}

# Add FRED only if it succeeded
fred_path = "../data/raw/india_repo_rate_raw.csv"
if os.path.exists(fred_path):
    files["India Repo Rate"] = fred_path

# Build summary
records = []
for name, path in files.items():
    info = get_file_info(path)
    info["Dataset"] = name
    info["Exists"]  = "✅" if os.path.exists(path) else "❌"
    records.append(info)

summary_df = pd.DataFrame(records, columns=[
    "Dataset", "Exists", "rows", "cols", "start", "end", "nulls", "size_kb"
])
summary_df.columns = [
    "Dataset", "File", "Rows", "Cols", "Start Date", "End Date", "Nulls", "Size (KB)"
]

print("=" * 75)
print("DATA INVENTORY — End of Day 4")
print("=" * 75)
print(summary_df.to_string(index=False))
print("=" * 75)

DATA INVENTORY — End of Day 4
         Dataset File  Rows  Cols Start Date   End Date  Nulls  Size (KB)
    USDINR daily    ✅  1302     6        N/A        N/A      5      107.9
    USDBRL daily    ✅  1302     6        N/A        N/A      5      109.3
INRBRL synthetic    ✅  1300     4 2021-03-04 2026-03-04      0       84.3
   USDINR hourly    ✅ 10549     6        N/A        N/A      5     1025.7
   USDBRL hourly    ✅  8416     6        N/A        N/A      5      829.8
 India Repo Rate    ✅    85     2 2019-01-01 2026-01-01      0        1.4
